1) Create conda or venv environment on head node <br>
2) Activate environment <br>
3) Install required packages (scanpy) + jupyterlab <br>
4) Submit interactive job from head node: <br>
    ```bsub -G team361 -q teramem -n4 -M 400000 -R"select[mem>400000 ] rusage[mem=400000 ]" -Is /bin/bash``` <br>
5) Activate environment <br>
6) Run  <br>
    ```jupyter-lab --no-browser --ip=0.0.0.0 --port=8888 &``` <br>
7) Forward port on local machine: <br>
    ```ssh -o TCPKeepAlive=no -o ServerAliveInterval=15 -L 8882:b11-hm-01:8888 sb75@farm22-head1.internal.sanger.ac.uk``` <br>
8) Open browser and go to ```http://localhost:8882/```

In [ ]:
import scanpy as sc
print(sc.__version__)

In [ ]:
artifacts_folder_path = '../../artifacts'
silver_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver'

load_timestamp = '06032025_085433_744' # model load timestamp
read_checkpoint = 'ep2' # model read checkpoint

In [ ]:
adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/adata_{read_checkpoint}_1000-1007-1008_combined.h5ad')

In [ ]:
adata # 11,036,633 cells

In [ ]:
adata.obs.tail(1)

In [ ]:
# Cell embedding 
adata.obsm['cell_emb_layer_6']

In [ ]:
# Neighborhood embedding
adata.obsm['neighborhood_emb_layer_6']

In [ ]:
adata.obs[['dataset', 'batch', 'cell']] = [s.split('_') for s in adata.obs.index]
adata.obs['dataset_batch'] = adata.obs['dataset'].astype(str) + adata.obs['batch'].astype(str)
adata.obs['cell'] = adata.obs['cell'].astype(int)

In [ ]:
# Perform stratified subsampling based on batch
sample_frac = 0.01 # 1 percent subsample
random_state = 42
sampled_df = adata.obs.groupby(['dataset', 'batch'], group_keys=False).apply(
        lambda x: x.sample(frac=sample_frac, replace=False, random_state=random_state))
adata_subset = adata[sampled_df.index]
del(sampled_df)

label = 'tissue'
# Neighborhood embedding UMAP
sc.pp.neighbors(adata_subset,
                n_neighbors=15,
                use_rep='neighborhood_emb_layer_6',
                key_added='neighborhood')
sc.tl.umap(adata_subset,
           neighbors_key='neighborhood',
           key_added='neighborhood_umap')
adata_subset.obsm['X_umap'] = adata_subset.obsm['neighborhood_umap']
sc.pl.umap(adata_subset,
           neighbors_key='neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata_subset,
                n_neighbors=15,
                use_rep='cell_emb_layer_6',
                key_added='cell')
sc.tl.umap(adata_subset,
           neighbors_key='cell',
           key_added='cell_umap')
adata_subset.obsm['X_umap'] = adata_subset.obsm['cell_umap']
sc.pl.umap(adata_subset,
           neighbors_key='cell',
           color=label)

In [ ]:
# Dataset ID 1000: eczema Lloyd
# Dataset ID 1007: kidney Daniyal
# Dataset ID 1008: embryo Adam

# Kidney raw data
# xhk1007-CV1-41b_1p
# xhk1007-CV2-41b_1p
# xhk1007-CV3-41b_1p
# xhk1007-CV4-41b_1p
# xhk1007-CV5-41b_1p
# xhk1007-CV6-41b_1p
# xhk1007-CV7-41b_1p
# xhk1007-CV8-41b_1p
# xhk1007-CV9-41b_1p

adata_raw = sc.read_h5ad(f'{silver_folder_path}/xhk1007-CV1-41b_1p/CV1-KID-0-FO-1.h5ad')

In [ ]:
# `original_cell_id` column

#'sample_batch_map': {
#    'CV1-KID-0-FO-1': 36,
#    'CV1-KID-0-FT-1': 14,
#    'CV1-KID-0-FT-2': 20,
#    'CV1-KID-0-FT-3': 1,
#    'CV2-KID-0-FO-1': 17,
#    'CV2-KID-0-FO-2': 33,
#    'CV2-KID-0-FT-1-s2': 4,
#    'CV2-KID-0-FT-2': 2,
#    'CV3-KID-0-FT-1-s3': 21,
#    'CV3-KID-0-FT-2-s3': 28,
#    'CV3-KID-0-FT-3-s3': 35,
#    'CV3-KID-0-FT-4-s3': 8,
#    'CV3-KID-0-FT-5': 16,
#    'CV4-KID-0-FO-1': 37,
#    'CV4-KID-0-FT-1-s3': 9,
#    'CV4-KID-0-FT-2-s3': 23,
#    'CV4-KID-0-FT-3-S3': 40,
#    'CV4-KID-0-FT-4-S3': 10,
#    'CV5-KID-0-FO-1': 31,
#    'CV5-KID-0-FT-1-s3': 38,
#    'CV5-KID-0-FT-2-s3': 5,
#    'CV5-KID-0-FT-4-S3': 7,
#    'CV6-KID-0-FO-1': 0,
#    'CV6-KID-0-FT-1': 27,
#    'CV6-KID-0-FT-2-s3': 3,
#    'CV6-KID-0-FT-3-s3': 12,
#    'CV6-KID-0-FT-4-S3': 19,
#    'CV6-KID-0-FT-5-S3': 24,
#    'CV7-KID-0-FO-1': 26,
#    'CV7-KID-0-FT-1-s3': 29,
#    'CV7-KID-0-FT-2-s3': 32,
#    'CV7-KID-0-FT-3-S3': 15,
#    'CV7-KID-0-FT-4-S3': 13,
#    'CV8-KID-0-FO-1': 18,
#    'CV8-KID-0-FT-1-s3': 11,
#    'CV9-KID-0-FO-1': 6,
#    'CV9-KID-0-FT-1-s3': 25,
#    'CV9-KID-0-FT-2-S3': 34,
#    'CV9-KID-0-FT-3-S3': 30,
#    'CV9-KID-0-FT-4-S3': 39,
#    'CV9-KID-0-FT-5-S4': 22}

In [ ]:
adata_raw.obs.head(1)

In [ ]:
import anndata as ad
import os

dataset_ids = ['1007']
adata_raw_list = []

# Load raw data
for subdir, _, files in os.walk(f'{silver_folder_path}/hst_corpus_70m'):
    if any(dataset_id in subdir.split('/')[-1].split('-')[0] for dataset_id in dataset_ids):
        print(f'Loading AnnData objects from {subdir}.')
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata_raw = sc.read_h5ad(file_path)
                adata_raw_list.append(adata_raw)     
concatenated_adata_raw = ad.concat(adata_raw_list, join='outer', index_unique=None)
concatenated_adata_raw.obs.index = concatenated_adata_raw.obs['cell_id']
del(concatenated_adata_raw.obs['cell_id'])

# Sort raw data
kidney_adata = adata[adata.obs['dataset'].apply(lambda x: x in dataset_ids)]
concatenated_adata_raw = concatenated_adata_raw[kidney_adata.obs_names, :].copy()

# Add embeddings
concatenated_adata_raw.obsm['cell_emb_layer_6'] = kidney_adata.obsm['cell_emb_layer_6']
concatenated_adata_raw.obsm['neighborhood_emb_layer_6'] = kidney_adata.obsm['neighborhood_emb_layer_6']

# Add metadata from original object
concatenated_adata_raw.obs['original_cell_id'] = concatenated_adata_raw.obs['batch'].astype(str) + concatenated_adata_raw.obs['original_cell_id'].astype(str)
daniyal_adata = sc.read_h5ad('/nfs/team361/dj17/RCC_nicheJEPA/inputs/analysis/RCC_20samples_afterclustering.h5ad')
daniyal_adata.obs['original_cell_id'] = daniyal_adata.obs['batch'].astype(str) + daniyal_adata.obs['cell_id'].astype(str)
obs_cols = [
    'original_cell_id',
    'level_1_cell_type',
    'level_2_cell_type',
    'level_3_cell_type'
]
daniyal_adata.obs = daniyal_adata.obs[obs_cols]
concatenated_adata_raw.obs = concatenated_adata_raw.obs.merge(daniyal_adata.obs, on='original_cell_id', how='left')

# Write adata
concatenated_adata_raw.write(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/adata_{read_checkpoint}_1007_annotated.h5ad')

In [ ]:
adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/adata_{read_checkpoint}_1007_annotated.h5ad')

In [ ]:
cell_type_key = 'level_1_cell_type'
spot_size = 20

In [ ]:
batch_ids = adata.obs['batch'].unique().tolist()

In [ ]:
adata_batch = adata[adata.obs['batch'] == 'CV1-KID-0-FO-1']

In [ ]:
batch_ids

In [ ]:
sc.pl.spatial(adata_batch,
              color=cell_type_key,
              #palette=custom_palette,
              spot_size=spot_size,
              #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
             )

In [ ]:
adata_batch = adata[adata.obs['batch'] == 'CV1-KID-0-FO-1']

In [ ]:
sc.pl.spatial(adata_batch,
              color=cell_type_key,
              #palette=custom_palette,
              spot_size=spot_size,
              #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
             )

In [ ]:
import os

In [ ]:
dataset_ids = ['1007']

In [ ]:


# Load raw data
for subdir, _, files in os.walk(f'{silver_folder_path}/hst_corpus_70m'):
    if any(dataset_id in subdir.split('/')[-1].split('-')[0] for dataset_id in dataset_ids):
        print(f'Loading AnnData objects from {subdir}.')
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                file_path = os.path.join(subdir, file)
                adata_raw = sc.read_h5ad(file_path)
                print(adata_raw.obs.head(1))
                print(adata_raw)

In [ ]:
adata_raw